# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and process the FAIR\(^2\) dataset using the `mlcroissant` library.

### Dataset Source
The dataset is described by a Croissant schema, available at the following URL:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Ensure `mlcroissant` is installed in the environment
!pip install -U mlcroissant

## 1. Data Loading
Load the dataset metadata using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Print high-level metadata
meta = dataset.metadata
print(f"{meta.name} ({meta.identifier})\n")
print(meta.description)

## 2. Data Overview
Explore available record sets and their fields, referencing all by their `@id`.

In [ ]:
# List available record sets by @id
from pprint import pprint

record_set_metadatas = list(dataset.metadata.record_sets)
print(f"Number of record sets: {len(record_set_metadatas)}\n")
for idx, rs in enumerate(record_set_metadatas):
    print(f"[{idx}] Record Set: {rs.name} (@id={rs.id})")
    if hasattr(rs, 'fields'):
        print("    Fields:")
        for f in rs.fields:
            print(f"      - {f.name} (@id={f.id}) type={f.data_type}")
    print()

# Save record set @ids for next steps
record_set_ids = [rs.id for rs in record_set_metadatas]

## 3. Data Extraction
Load each record set via its `@id` into a pandas DataFrame for analysis.

We'll show field `@id`s for clarity and reproducibility.

In [ ]:
# Extract to DataFrames
dfs = {}
for rset_id in record_set_ids:
    print(f"Loading record set: {rset_id}")
    records = list(dataset.records(record_set=rset_id))  # record_set argument expects the @id
    df = pd.DataFrame(records)
    dfs[rset_id] = df
    print(f" - {df.shape[0]} rows, {df.shape[1]} cols: {list(df.columns)}\n")

# Choose a main record set for exploration
main_set_id = record_set_ids[0] if record_set_ids else None
df = dfs[main_set_id] if main_set_id else None

if df is not None:
    print(f"First 5 rows of record set '@id={main_set_id}':")
    display(df.head())
else:
    print("No data extracted.")

## 4. Exploratory Data Analysis (EDA)
We will examine a numeric field in the main record set. All references are declared using their `@id`s.

In [ ]:
import numpy as np

if df is not None and not df.empty:
    # Heuristically pick a numeric (float or integer) field from metadata
    numeric_field_obj = None
    group_field_obj = None
    rs_meta = next((rs for rs in record_set_metadatas if rs.id == main_set_id), None)
    for field in rs_meta.fields:
        if field.data_type in ('Float', 'Integer', 'Number') and numeric_field_obj is None:
            numeric_field_obj = field
        if group_field_obj is None and field.data_type == 'Text':  # pick a likely group field
            group_field_obj = field

    if numeric_field_obj is not None:
        numeric_field_id = numeric_field_obj.id
        colname = numeric_field_id
        if colname in df.columns:
            # Drop NAs if any
            values = df[colname].apply(pd.to_numeric, errors='coerce')
            threshold = values.mean()  # Example threshold
            filtered_df = df[values > threshold].copy()
            print(f"Filtered records (by '@id={colname}') > mean (threshold = {threshold:.2f}): {filtered_df.shape[0]} records.")
            # Normalize field
            filtered_df[f"{colname}_normalized"] = (values - values.mean()) / values.std()
            display(filtered_df[[colname, f"{colname}_normalized"]].head())

            # Group by a text/categorical field
            if group_field_obj is not None:
                group_col = group_field_obj.id
                if group_col in df.columns:
                    grouped_df = filtered_df.groupby(group_col)[colname].mean().reset_index()
                    print(f"Grouped mean of field '@id={colname}' by '@id={group_col}':")
                    display(grouped_df.head())
        else:
            print(f"Numeric field '@id={colname}' not found in columns.")
    else:
        print("No numeric field detected.")
else:
    print("No data for EDA.")

## 5. Visualization
Show visualization of a numeric field distribution (if any found) and groupings using matplotlib and seaborn.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if df is not None and not df.empty and numeric_field_obj is not None and numeric_field_obj.id in df.columns:
    colname = numeric_field_obj.id
    plt.figure(figsize=(8,4))
    sns.histplot(df[colname].apply(pd.to_numeric, errors='coerce').dropna(), kde=True)
    plt.title(f"Distribution of {numeric_field_obj.name} (@id={colname})")
    plt.xlabel(numeric_field_obj.name)
    plt.show()

    # If grouped
    if group_field_obj is not None and group_field_obj.id in df.columns:
        plt.figure(figsize=(10,4))
        sns.boxplot(x=df[group_field_obj.id], y=df[colname].apply(pd.to_numeric, errors='coerce'))
        plt.title(f"{numeric_field_obj.name} grouped by {group_field_obj.name}")
        plt.xlabel(group_field_obj.name)
        plt.ylabel(numeric_field_obj.name)
        plt.xticks(rotation=45)
        plt.show()
else:
    print("No suitable numeric field found for visualization.")

## 6. Conclusion
In this notebook, we explored the FAIR\(^2\) dataset's structure and demonstrated how to reference and analyze its entities by `@id` using the `mlcroissant` library. Using `mlcroissant`, you can generalize these workflows to any dataset described by a Croissant schema, ensuring reproducibility and FAIR principles.